<a href="https://colab.research.google.com/github/HenryZumaeta/py4dl_EPC2026/blob/main/C05_Script01_DescargaKaggle_Regularizadores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Punto de partida para la sesion 5 (https://gist.github.com/robintux/841abf0dd322690f220a34fb4ac265ff)


# modulos
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn
from sklearn.model_selection import train_test_split
from sklearn import metrics

# Redes Neuronales: Keras
from  keras.models import Sequential
from keras.layers import Dense

# Descargamos el dataset de kaggle
!kaggle datasets download robintux/salud-mental-agotamiento-estudiantil

# Descomprimirlo
!unzip salud-mental-agotamiento-estudiantil.zip

# Cargar en memoria
df = pd.read_csv("Salud_mental_agotamiento_estudiantil.csv")
df = df.drop("Unnamed: 0", axis = 1)

df = df[df.dropout_risk >= 0.1].reset_index(drop = True)
df = df.sample(frac= 0.1).reset_index(drop = True)

X = df.iloc[:, :-1]
y = df.dropout_risk

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.25, random_state=666)
print(f"   NUmero de Observaciones (Train) : {X_train.shape[0]:d}")
print(f"   NUmero de Observaciones (Train) : {X_test.shape[0]:d}")

# Necesitamos modfificar esta rutina para lograr un codigo multiplataforma, mas escalable
# y que no dependa de una shell interactiva (IPython)

# Recomendacion : Aprender todo lo concerniente a  los magic command (built-ins)

Dataset URL: https://www.kaggle.com/datasets/robintux/salud-mental-agotamiento-estudiantil
License(s): apache-2.0
100% 130M/130M [00:00<00:00, 140MB/s]

Archive:  salud-mental-agotamiento-estudiantil.zip
  inflating: Salud_mental_agotamiento_estudiantil.csv  
   NUmero de Observaciones (Train) : 53676
   NUmero de Observaciones (Train) : 17892


In [ ]:
# El primer trabajo es acceder a los datos : API de Kaggle
# https://www.kaggle.com/settings/api

In [2]:
# Practicas Estandar
# Gestion del sistema de archivos en python
import os
import sys
import shutil
from pathlib import Path

# Modulos Cientificos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn import metrics

# Keras
from keras.models import Sequential
from keras.layers import Dense


# Variables (Constantes) de configuracion
# Es practica comun que las constantes se escriban en mayusculas
DATASET_NAME = "Salud_mental_agotamiento_estudiantil.csv"
DATASET_SLUG = "robintux/salud-mental-agotamiento-estudiantil"

# Necesitamos determinar si nuestro codigo se esta ejecutando en un entorno interractivo (IPython)
IS_JUPYTER = hasattr(sys, "ps1") and "ipykernel" in sys.modules

# PEP 484 : Type Hints en Python
# El modulo typing : (ESTUDIAR)
    # Colecciones : List, Dict, Set, Tuple
    # Opcionales y Uniones : Optional[T], Union[T1, T2]
    # Callable y protocolos
    # Tipos especiales : Any, NoReturn, ClassVar
# El PEP484 definio una interfaz clara que impulso un ecosistema de verificacion estatica
    # mypy
    # pyright/pylance
    # ruff
    # IDE : Pycharm, VSCODE, Spyder y jupyter

# Necesitamos implementar una funccion que permita una salida controlada compatible
# con scripts (archivos de extension .py) y notebooks (archivos de extension ipynb)
def safe_exit(message: str, code: int = 1):
    """
    Salida controlada compatible con interpretes : python e ipython
    """
    if IS_JUPYTER:
        raise RuntimeError(message)
    else:
        print(message, file = sys.stderr)
        sys.exit(code)

# Necesitamos un funcion que implemente un mecanismo flexible de busqueda y verificacion
# de archivos
def file_exists(filename : str , search_dirs:list[str] = None) -> Path | None :
    """
    Funcion que valida la existencia del dataset
    """
    if search_dirs is None:
        search_dirs = [".", "./data", "./datasets"]

    for directory in search_dirs:
        file_path = Path(directory)/filename
        if file_path.exists() and file_path.is_file():
            return file_path.resolve()

    for file_path in Path(".").rglob(filename):
        if file_path.is_file():
            return file_path.resolve()
    return None

# Necesitamos una funcion que descargue y descomprima el dataset a trabajar
# En esta funcion podemos tener dos principales problemas
    # El modulo kaggle no este instalado
    # Problemas con la descarga
def download_dataset(slug : str, output_dir :str = ".") -> bool :
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        api.dataset_download_files(slug, path = output_dir, unzip = True, quiet = False)
        return True
    except ImportError:
        print("El modulo kaggle no esta instalado")
        print("Instala con : pip install kaggle")
        return False
    except Exception as e :
        print(f"Error en la descarga : {e}")
        return False

# Necesitamos una funcion que utilice las funciones file_exists  y download_dataset
def ensure_dataset(filename : str, slug : str) -> Path :
    csv_path = file_exists(filename)
    if csv_path :
        print(f"Dataset encontrada {csv_path}")
        return csv_path

    print(f"Archivo : {filename} no encontrado .")
    print("... llamando a las rutinas para descargar el archivo desde kaggle")

    if not download_dataset(slug):
        safe_exit(
            "No se pudo descargar el dataset \n"
            "Verifica :\n"
            "   1. pip install kaaggle\n"
            "   2. Configuracion del token\n"
            "   3. Permisos del archivo que almacena el token"
        )
    csv_path = file_exists(filename)
    if csv_path:
        return csv_path

    # Fallbaccck : Buscar cualquier csv reciente
    candidates = [f for f in Path(".").glob("*.csv") if "Salud_mental" in f.name or "salud" if f.name.lower() ]
    if candidates:
        return max(candidates, key = lambda p: p.stat().st_mtime).resolve()

    safe_exit(f"Error : No se encontro {filename} tras la descarga")

In [4]:
# Flujo de trabajo
try:
    # Configuracion de la API de kaggle
    print("Configurando kaggle")
    os.system("<Configurar de Kagle>")
    print("Fin de la conf. de kaggle")

    # Garantizar el dataset
    print("llamando a ensure_dataset")
    csv_path = ensure_dataset(DATASET_NAME, DATASET_SLUG)
    print("terminando ensure_dataset")

    # Cargamos el dataset
    print("")
    print(f"cargando : {csv_path.name}")
    df = pd.read_csv(csv_path)

    # Eliminemos la columna "Unnamed: 0"
    if "Unnamed: 0" in df.columns:
        df = df.drop("Unnamed: 0", axis = 1)

    # Verifiquemos que la variable dependiente este en el dataframe
    if "dropout_risk" not in df.columns:
        safe_exit(f"La variable dependiente 'dropout_risk' no existe. Columnas : {list(df.columns)}")

    # Preparacion de los datos para el modelado
    df = df[df.dropout_risk >= 0.1].reset_index(drop = True)
    df = df.sample(frac= 0.1).reset_index(drop = True)

    X = df.iloc[:, :-1]
    y = df.dropout_risk

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.25, random_state=666)
    print(f"Número de Observaciones (Train) : {X_train.shape[0]:d}")
    print(f"Número de Observaciones (Test) : {X_test.shape[0]:d}")

except Exception as e:
    print(f"\n ERROR CRITICO : {type(e).__name__} : {e}")


Configurando kaggle
FIN de la conf. de kaggle
llamando a ensure_dataset
Dataset encontrada /content/Salud_mental_agotamiento_estudiantil.csv
terminando ensure_dataset

cargando : Salud_mental_agotamiento_estudiantil.csv
Número de Observaciones (Train) : 53676
Número de Observaciones (Test) : 17892


# Ultima version de build_mlp_regression

```python

def build_mlp_regression(
        input_dim,
        hidden_units = [512,128,96,64,48,16,4],
        kernel_initializer = "he_normal",
        bias_initializer = "zeros"):
    ModelBase = Sequential()
    for i, units in enumerate(hidden_units):
        ModelBase.add(Dense(
            units = units,
            activation = "relu",
            input_dim = input_dim if i == 0 else None,
            name = f"capa_oculta_{i+1}",
            kernel_initializer = kernel_initializer,
            bias_initializer = bias_initializer
        ))
    ModelBase.add(Dense(
        units = 1 ,
        activation ="linear",
        name = "Capa_de_salida",
        kernel_initializer = kernel_initializer,
        bias_initializer = bias_initializer
    ))

    return ModelBase

# Experimentacion con diferentes arquitecturas
modelo_a = build_mlp_regression(input_dim=X_train.shape[1], hidden_units = [512,256,128])
modelo_b = build_mlp_regression(input_dim=X_train.shape[1], hidden_units = [16,24,32,48])
modelo_c = build_mlp_regression(input_dim=X_train.shape[1], hidden_units = [256,256,256])

```

In [5]:
# Necesitamos algunas capas  y regularizadores
import keras
from keras.layers import Dropout, BatchNormalization
from keras.regularizers import l1, l2

def build_mlp_regression(config : dict):
    # 1. Extraer la configuracion de argumentos para las clases de keras autilizar
    input_dim = config.get("input_dim")
    hidden_units = config.get("hidden_units", [128,64, 32])
    activations = config.get("activations", "relu")
    # Capas Dropout
    dropout_rates = config.get("dropout_rates", 0.0)
    # Capa de Normalizacion
    use_batchnorm = config.get("use_batchnorm", False)
    kernel_regularizer = config.get("kernel_regularizer", None)
    kernel_initializer = config.get("kernel_initializer", "he_normal")
    bias_initializer = config.get("bias_initializer", "zeros")
    seed = config.get("seed", 666)
    compile_model = config.get("compile", True)
    optimizer = config.get("optimizer", keras.optimizers.Adam(learning_rate = 0.001))
    loss = config.get("loss", "mse")
    metrics = config.get("metrics", ["mae"])
    # Esto puede aumentar ...

    # 2. Reproduibilidad
    keras.utils.set_random_seed(seed)

    # 3. Normalizacion de parametros por capa
    if isinstance(activations, str):
        activations = [activations] * len(hidden_units)
    if isinstance(dropout_rates, (int, float)):
        dropout_rates = [dropout_rates] * len(hidden_units)

    # El usuario puede proveer estos argumentos (activations y/o dropout_rates) como listas
    # pero que no tienen la misma cantidad de elementos
    if not (len(hidden_units) == len(activations) == len(dropout_rates)):
        raise ValueError("Las listas : 'hiddent_units', 'activations'  y 'dropout_rates' deben tener la misma longitud")

    # 4. Construcion del modelo
    model = Sequential(name = "MLP_REGresion")
    for i,(units,act,drop) in enumerate(zip(hidden_units, activations, dropout_rates)):
        layer_kwargs = dict(
            units = units,
            activation = act,
            kernel_initializer = kernel_initializer,
            bias_initializer = bias_initializer,
            kernel_regularizer = kernel_regularizer,
            name = f"dense:_{i+1}"
        )

        if i == 0:
            layer_kwargs["input_shape"] = (input_dim, )

        model.add(Dense(**layer_kwargs))

        if use_batchnorm:
            model.add(BatchNormalization(name = f"batchnorm_{i+1}"))
        if drop > 0:
            model.add(Dropout(rate = drop, name = f"dropout_{i+1}"))

    # Capa de Salida
    model.add(Dense(
        units = 1,
        activation = "linear",
        kernel_initializer = kernel_initializer,
        bias_initializer = bias_initializer,
        kernel_regularizer = kernel_regularizer,
        name = "capa_De_salida"
    ))

    # 5. Compilacion opcional
    if compile_model:
        model.compile(
            optimizer = optimizer,
            loss = loss,
            metrics = metrics
        )

    # 6. Retornamos modelo + configuracion  (util para logging y tracking)
    return model, config

In [6]:
# Definamos una lista de configuraciones
configs = [
    {"hidden_units":[256,128], "dropout_rates" :[0.2, 0.1], "use_batchnorm" : False},
    {"hidden_units":[256,128], "dropout_rates" :[0.2, 0.1], "use_batchnorm" : True},
    {"hidden_units":[64,32,16], "dropout_rates" :0.0 , "activations" :  ["selu", "relu", "gelu"]}
]

# Ejecucion
models_list = []

for i, config in enumerate(configs) :
    print(f"\n--- Construyendo modelo : {i+1} ---")
    try:
        config["input_dim"] = X.shape[1]
        model, used_config = build_mlp_regression(config)

        # Imprimamos un resumen para verificar la arquitectura
        model.summary()

        # Guardamos el modelo
        models_list.append(model)

    except Exception as e:
        print(f"Error al construir el modelo : {i+1} : {e} ")


--- Construyendo modelo : 1 ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "MLP_REGresion"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense:_1 (Dense)                │ (None, 256)            │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense:_2 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ capa_De_salida (Dense)          │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,377 (146.00 KB)

...

Model: "MLP_REGresion"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense:_1 (Dense)                │ (None, 64)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense:_2 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense:_3 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ capa_De_salida (Dense)          │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,713 (14.50 KB)

 Trainable params: 3,713 (14.50 KB)

 Non-trainable params: 0 (0.00 B)